<a href="https://colab.research.google.com/github/apopodko/Yandex.Praktikum-DS-Projects/blob/master/text_image_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Поиск по изображениям

Вы работаете в фотохостинге для профессиональных фотографов «Со Смыслом» (“With Sense”). Ваши пользователи размещают свои фотографии на хостинге и сопровождают их полным описанием: указывают место съёмок, модель камеры и т. д. Ваш отдел занимается экспериментом по разработке поиска референсных фотографий для фотографов. Суть поиска заключается в следующем: пользователь сервиса вводит описание нужной сцены. Сервис выводит несколько фотографий с такой же или похожей сценой.

Чтобы эксперимент получил право на жизнь, нужно защитить его перед руководителем компании. Для защиты необходимо презентовать так называемый PoC (Proof of Concept, Проверка концепции) — продемонстрировать, что такой проект практически осуществим. Вам поручено разработать демонстрационную версию поиска изображений по запросу.

## Описание проекта

**Заказчик**
Фотохостинг для профессиональных фотографов «Со Смыслом»


**Входные данные**
Изображения, идентификаторы описания, тексты описанияб оценки экспертов и краудсорсинга по соответствию описания и изображения


**Задача**
Исследовать данные, определить и исключить изображения, подпадающие под юридические ограничения. Создать модель по поиску изображения, соответствующего описаниию запроса

**Цель**
Разработать демонстрационную версию поиска изображений по запросу в качестве PoC для презентации заказчику

**Описание входных данных**

**Путь к входным данным прописан ниже в константах `DATA_PATH`**
- `train_dataset.csv` - тренировочный датасет, содержащий имя файла изображения, идентификатор описания и текст описания
    - `image` - имя файла изображения
    - `query_id` - идентификатор описания
    - `query_text` - текстовое описание
    
    
- `test_queries.csv` - тестовый датасет с идентификатором запроса, текстом запроса и релевантными изображениями


- `CrowdAnnotations.tsv` - данные по соответствию изображения и описания, полученные с помощью краудсорсинга
    - `image` - имя файла изображения
    - `query_id` - идентификатор описания
    - `match_score` - доля людей, подтвердивших, что описание соответствует изображению
    - `pos_match_count` - количество человек, подтвердивших, что описание соответствует изображению
    - `neg_match_count` - количество человек, подтвердивших, что описание не соответствует изображению
    
    
- `ExpertAnnotations.tsv` - данные по соответствию изображения и описания, полученные в результате опроса экспертов
    - `image` - имя файла изображения
    - `query_id` - идентификатор описания
    - `expert_val_1` - оценка первого эксперта соответствия изображения описанию
    - `expert_val_2` - оценка второго эксперта соответствия изображения описанию
    - `expert_val_3` - оценка третьего эксперта соответствия изображения описанию
    
    
- `train_images` - папка с тренировочной выборкой изображений
- `test_images` - папка с тестовой выборкой изображений

In [ ]:
!pip install keras==3.8.0
!pip install matplotlib==3.10.0
!pip install numpy==2.0.2
!pip install pandas==2.2.2
!pip install scikit-learn==1.6.1
!pip install seaborn==0.13.2
!pip install spacy==3.8.5
!pip install tensorflow[and-cuda]==2.18.0
!pip install tqdm==4.67.1
!pip install transformers==4.51.1

In [ ]:
import pandas as pd
import numpy as np
import os
from collections import Counter
from tqdm.notebook import tqdm, trange
import re
import random

import matplotlib.pyplot as plt
from textwrap import wrap
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GroupShuffleSplit

import tensorflow as tf
from tensorflow.keras.applications.inception_resnet_v2 import (
    InceptionResNetV2,
    preprocess_input
)
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Lambda
from tensorflow.keras import Model, Input
from tensorflow.keras import layers, models
from PIL import Image
import spacy
from transformers import (
    BertTokenizer,
    TFBertModel,
    ViTFeatureExtractor,
    TFViTModel,
    CLIPProcessor,
    TFCLIPModel
)
from collections import defaultdict
DATA_PATH = '/kaggle/input/image-text-retrieve/dsplus_integrated_project_4/to_upload/'
TRAIN_IMAGES_DIR = 'train_images/'
TEST_IMAGES_DIR = 'test_images/'
SEED = 42
RANDOM_STATE = 42
tf.random.set_seed(12345)
np.random.seed(12345)
tqdm.pandas()

## Импорт данных и разведочный анализ

In [ ]:
def analyze(data):

    print('Общая информация о признаках\n')
    data.info()

    print('\nПервые пять строк датасета')
    display(data.head())

    print('\nСтатистика по признакам')
    display(data.describe())

    stats = {'values' : [
                         'Unique, count',
                         'Unique, %',
                         'Missing, count',
                         'Missing, %'
                        ]}
    for col in data:
        stats[col] = [
                      data[col].nunique(),
                      round(100 * data[col].nunique() / data.shape[0], 2),
                      data[col].isnull().sum(),
                      round(100 * data[col].isnull().sum() / data.shape[0], 2)
                     ]
    stats = pd.DataFrame(stats).set_index('values').T
    print('\nСтатистика по уникальным значениям и пропускам')
    display(stats.sort_values('Missing, %', ascending=False))

    print(f'\nПолных явных дубликатов - {data.duplicated().sum()}')
    print('\nУникальные значения')
    for col in data:
        if data[col].nunique() < 15:
            print(f'Признак {col}, значения:', np.sort(data[col].unique()))

Импортируем все таблицы

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_PATH, 'train_dataset.csv'))

df_crowd = pd.read_csv(os.path.join(DATA_PATH, 'CrowdAnnotations.tsv'),
                       sep='\t',
                       header=None,
                       names=['image',
                              'query_id',
                              'crowd_mean_score',
                              'crowd_pos_count',
                              'crowd_neg_count'])

df_expert = pd.read_csv(os.path.join(DATA_PATH, 'ExpertAnnotations.tsv'),
                        sep='\t',
                        header=None,
                        names=['image',
                               'query_id',
                               'expert_val_1',
                               'expert_val_2',
                               'expert_val_3'])

df_test = pd.read_csv(os.path.join(DATA_PATH, 'test_queries.csv'), index_col=0,
                     sep='|')

### `train_dataset`

In [ ]:
analyze(df_train)

- 5822 записи
- Пропусков нет
- Полных дубликатов нет
- Уникальных элементов по признакам
    - `image` - 1000 изображений
    - `query_id` - 977 айди запросов
    - `query_text` - 977 текстов запросов
- У всех `query_id` стоит #2, на каждую картинку может быть больше 5 разных запросов

### `CrowdAnnotations`

In [ ]:
analyze(df_crowd)

- 47830 записей
- Пропусков нет
- Полных дубликатов нет
- Уникальных элементов по признакам
    - `image` - 1000 изображений
    - `query_id` - **1000** айди запросов, отличается от `train_dataset`
    - `crowd_mean_score` - 12 степеней средней оценки от 0 до 1 включительно
    - `crowd_pos_count` - целые от 0 до 5, то есть максимально 5 человек проголосовало за соответствие
    - `crowd_neg_count` - целые от 0 до 6, то есть максимально **6** человек проглосовало за несоответствие

### `ExpertAnnotations`

In [ ]:
analyze(df_expert)

- 5822 записи
- Пропусков нет
- Полных дубликатов нет
- Уникальных элементов по признакам
    - `image` - 1000 изображений
    - `query_id` - 977 айди запросов, так же как и в `train_dataset`
    - `expert_val_1` - натуральные от 1 до 4 включительно
    - `expert_val_2` - натуральные от 1 до 4 включительно
    - `expert_val_3` - натуральные от 1 до 4 включительно

### `test_queries`

In [ ]:
analyze(df_test)

- 500 записей
- Пропусков нет
- Полных дубликатов нет
- Уникальных элементов по признакам
    - `image` - 100 изображений
    - `query_id` - 500 айди запросов
    - `query_text` - 500 текстов запросов

### `train_images` и `test_images`

In [ ]:
print('Количество изображений в папке train_images - ',
      len(next(os.walk(os.path.join(DATA_PATH, 'train_images')))[2]))
print('Количество изображений в папке test_images - ',
      len(next(os.walk(os.path.join(DATA_PATH, 'test_images')))[2]))

- В папке `train_images` - 1000 изображений
- А вот в папке `test_images` - 101 изображение, хотя в тестовой таблице уникальных названий изображений - 100!

### Вывод

- Импортировали все датасеты успешно, пропусков и полных дубликатов нет
- В каждом 1000 уникальных объектов изображений
- `df_train` и `df_expert` - 977 уникальных объектов запросов, `df_crowd` - 1000
- Получается, что в `df_train` есть изображения без подходящего запроса, либо запросы, к которым очень хорошо подходит несколько изображений
- Всего записей
    - `df_train` - 5822
    - `df_expert` - 5822
    - `df_crowd` - 47830
- Структура оценок
    - `df_expert` - 3 эксперта с оценками от 1 до 4 включительно
    - `df_crowd` -  12 степеней средней оценки от 0 до 1 включительно
        - `crowd_pos_count` - целые от 0 до 5, то есть максимально 5 человек проголосовало за соответствие
        - `crowd_neg_count` - целые от 0 до 6, то есть максимально **6** человек проглосовало за несоответствие
- По разведочному анализу видим, что у нас
    - во всех тренировочных датасетах по 1000 изображений
    - `df_expert` и `df_train` совпадают по количеству уникальных запросов - 977, и по количеству записей
    - `df_crowd` и `df_train` отличаются, 1000 против 977, возможно какие-то дополнительные запросовые описания были
    - В любом случае будем точно объединять `df_train` и `df_expert`, экспертные оценки чаще всего поточнее, чем краудсорс, но не факт, проверим позже
    - В оценках через краудсорс в разы больше пар изображение-запрос (в 8 раз), что в теории может дать лучший результат по метрикам, но точность разметки может быть хуже, причем настолько, что испортит экспертную оценку, тоже посмотрим
    - Объединение будем производить по полям `image` и `query_id`, так как на одну и то же изображение может приходиться несколько описаний и наоборот

## Исследовательский анализ

### `df_train` и `df_expert`

Посмотрим на 23 изображения, для которых нет описания

In [ ]:
images_list_ = df_train['image'].unique() + '#2'
set(images_list_).difference(set(df_train['query_id'].unique()))

In [ ]:
df_expert.query('image=="1343426964_cde3fb54e8.jpg"')

In [ ]:
df_expert.query('image=="1343426964_cde3fb54e8.jpg#2"')

- В принципе такое тоже может быть, что ни один запрос идеально не подходит для описания изображения, а подходит лишь приближенно, скорее всего оставим, но по ходу дела можно будет еще подумать
- Запроса `1343426964_cde3fb54e8.jpg#2` вообще нет
- Тем более, в краудсорсинговых оценках было полное и подходящее описание, возможно, если будем все-так объединять и с ним, то все будет в порядке
- Таблицы `df_train` и `df_expert`, как описывали в выводе, состоят из одинаковых пар изображение-запрос, проблем нет, поэтому объединим их

In [ ]:
df_expert[df_expert['image']+'#2'==df_expert['query_id']].head(7)

Так же отметим, что изображение полностью соответствует запросу, когда id запроса равно названию изображения + `#2`

In [ ]:
df_train = pd.merge(df_train, df_expert, on=['image', 'query_id'], how='inner')
df_train.info()

Да, наше предположение об одинаковости пар изображение-запрос для `df_train` и `df_expert` оказалось верным, внутренним объединением осталось все так же 5822 пары

In [ ]:
sns.set(font_scale=1.2)
sns.set_style('whitegrid', {'axes.edgecolor': 'black'})

In [ ]:
exp_array_ = ['expert_val_1',
              'expert_val_2',
              'expert_val_3']
(pd.DataFrame(df_train[exp_array_].value_counts())
 .sort_values(by=exp_array_)
 .plot(kind='bar', legend=False, figsize=(16, 8), fontsize=14, grid=True))
plt.xlabel('Оценки (Эксперт_1, Эксперт_2, Эксперт_3)', fontsize=14)
plt.ylabel('Количество запросов', fontsize=14)
plt.title('Распределение запросов по сгруппированным оценкам экспертов',
          fontsize=16);

- В целом эксперты более менее согласованны, хотя есть редкие случаи вроде (1, 3, 3) или (1, 4, 4) например
- Посмотрим на три стратегии аггрегации оценок
    - Среднее арифметическое, то есть все 3 эксперта равноценны и мы им одинаково доверяем
    - Мода - наиболее частая оценка среди всех экспертов, если все 3 оценки разные (3.5 % случаев) - возьмем медиану
    - Медиана - центральная из упорядоченного набора, если один эксперт зажестил например

In [ ]:
def mode_agg(ratings):
    counts = Counter(list(ratings))
    mode, count = counts.most_common(1)[0]
    if count == 1:
        return int(np.median(ratings))
    else:
        return mode

df_train['exp_mode_score'] = df_train[exp_array_].apply(mode_agg, axis=1)
df_train['exp_median_score'] = df_train[exp_array_].median(axis=1)
df_train['exp_mean_score'] = df_train[exp_array_].mean(axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
sns.histplot(df_train.loc[:, 'expert_val_1':'exp_mean_score'],
             stat='count',
             discrete=True,
             multiple='dodge',
             shrink=0.7,
             edgecolor='black',
             alpha=1)
plt.title('Распределение оценок соответствия запросов по экспертам',
          fontsize=16)
plt.xlabel('Оценка', fontsize=15)
plt.ylabel('Количество запросов', fontsize=15)
legend = ax.get_legend()
handles = legend.legend_handles
ax.legend(handles,
          ['expert_val_1',
           'expert_val_2',
           'expert_val_3',
           'mode',
           'median',
           'mean_ROUNDED']
         )
plt.xticks([1,2,3,4]);

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
sns.barplot(df_train['exp_mean_score'].value_counts(),
            edgecolor='black',
            alpha=1,
            ax=ax)
plt.title('Распределение средних оценок соответствия запросов',
          fontsize=16)
plt.xlabel('Оценка', fontsize=15)
plt.ylabel('Количество запросов', fontsize=15)
ax.set_xticklabels(['{:.2f}'.format(float(t.get_text())) for t in ax.get_xticklabels()]);

- Видим, что есть какое-то количество изображений около 100, где все эксперты дали разные оценки
- Эксперт_1 занижает оценки, эксперт_3 - наоборот, завышает, а эксперт_2 сбалансированно между ними расположен
- С другой стороны, аггрегированные оценки моды и медианы почти везде дают похожий результат, кроме 2, 0 и 3, видимо, когда у всех экспертов разные оценки медиана чаще 2, и редко 3, поэтому из этих двух можно использовать медиану, кроме того, визуально они похожи на целую часть от средней оценки по всем экспертам
- Если средние оценки считать правильно, то получается в целом тоже неплохая картина, сравним ее позже с оценками краудсорсинга и там выберем либо среднюю, либо медиану, но я бы склонялся к средней, так как она повторяет медиану, если брать целую часть, но при этом дает более точную оценку, выбросов у нас тут нет

In [ ]:
df_train.drop(exp_array_ + ['exp_mode_score'], axis=1, inplace=True)
del df_expert

In [ ]:
scaler = MinMaxScaler()
df_train['exp_median_score'] = (scaler
                                .fit_transform(df_train['exp_median_score']
                                               .values.reshape(-1, 1))
                                .round(2))
df_train['exp_mean_score'] = (scaler
                              .fit_transform(df_train['exp_mean_score']
                                             .values.reshape(-1, 1))
                              .round(2))

In [ ]:
df_train.head()

### `df_crowd`

Сразу объединим таким образом, чтобы остались только запросы, у которых есть текстовое описание

In [ ]:
df_crowd = df_crowd.merge(df_train[['query_id', 'query_text']].drop_duplicates(),
                          on='query_id',
                          how='inner')
df_crowd.info()

А здесь были запросы, для которых у нас нет текстового описания, целых 1109 штук (2.3 %), автоматом от них избавились

In [ ]:
df_crowd['crowd_mean_score'] = df_crowd['crowd_mean_score'].round(2)
df_crowd['crowd_mean_score'].value_counts(normalize=True).sort_index()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
sns.barplot(df_crowd['crowd_mean_score'].value_counts(),
             edgecolor='black',
             alpha=1, ax=ax)
plt.title('Распределение оценок краудсорсинга соответствия запросов',
          fontsize=16)
plt.xlabel('Оценка', fontsize=15)
plt.ylabel('Количество запросов', fontsize=15);

- Система оценок краудсорсинга в целом похожа на медианную у экспертов, есть небольшие вкрапления вне 4 основных групп, когда количество пользователей было больше 3
- У некоторых изображений было более 1 полностью подходящего запроса, по мнению краудсорса, 1305 (2.8 %) пар с оценкой 1
- Попробуем посмотреть на корреляцию оценок краудсорсинга и экспертов на одинаковых парах. Для сравнения возьмем медианную оценку экспертов

In [ ]:
df_compare = df_train.merge(df_crowd, on=['image', 'query_id'], suffixes=(None, '_'))
df_compare.drop('query_text_', axis=1, inplace=True)
df_compare.info()

In [ ]:
fig = plt.figure(figsize=(16, 8))
sns.countplot(
    data=df_compare,
    x='crowd_mean_score',
    hue='exp_median_score',
    stat='percent',
    palette='tab10',
    edgecolor='black',
    alpha=1
)
plt.title(
    'Распределение оценок краудсорсинга соответствия запросов '
    'в зависимости от медианной оценки экспертов',
    fontsize=16
)
plt.xlabel('Оценка краудсорсинга', fontsize=15)
plt.ylabel('Количество запросов, %', fontsize=15)
plt.legend(title='Медианная оценка экспертов');

- Всего для сравнения набралось 2329 пар, немного меньше 50 % датасета с оценками экспертов
- Краудсорсинг заметно занижает оценки соответствия пар по сравнению с экспертами, которым мы доверяем больше, со слов заказчика - они ж эксперты
- Встает выбор, объединять наш датасет с датасетом краудсорсинга
    - С одной стороны, почти в 10 раз пар запрос-изображение, то есть заметно вырастет обучающая выборка, но 88 % (около 42 тыс.) запросов полностью не подходят ни под одно изображение по мнению краудсорсинга, причем из этих запросов, а так же с более высокой оценкой какая-то часть может все-таки подходить больше к изображению, чем оценили краудсорсеры. То есть мы из желания лучше научить модель определять то, чего нет на изображении можем лишиться значительной её описательной силы на какие-то детали изображения, например на изображении `пасмурное или дождливое небо`, а в описании `облачное небо` и аутсорсеры с занижением оценки могли поставить 0 в итоге, а не 0.33 или 0.67, что сделает труднее связку неба в запросе и на изображении для модели, то же касается и положительных пар
    - С другой стороны, можно не использовать оценки краудсорсинга, да выборка будет меньше, но уверенность в качестве разметки будет сильно выше
- Посмотрим качественно на всю эту ситуацию, чтобы лучше понять разницу в оценках

In [ ]:
compare_datagen = ImageDataGenerator(rescale=1/255.)

def image_comparison(dataset):
    compare_flow = compare_datagen.flow_from_dataframe(
        dataset,
        directory=os.path.join(DATA_PATH,
                              'train_images'),
        x_col='image',
        y_col=['query_text',
               'exp_median_score',
               'exp_mean_score',
               'crowd_mean_score'],
        target_size=(256, 256),
        batch_size=32,
        shuffle = False,
        class_mode='raw',
        seed=SEED
    )

    compare_features, compare_target = next(compare_flow)

    fig = plt.figure(figsize=(16, 16))
    plt.suptitle('Примеры пар изображений, оценок и текстовых запросов\n')
    for i in range(9):
        fig.add_subplot(3, 3, i+1)
        plt.imshow(compare_features[i])
        plt.xticks([])
        plt.yticks([])
        plt.title(
            f'Краудсорс: {compare_target[i][3]:>9}'.rjust(45) +
            f'\nЭксперты (медиана|средняя): {compare_target[i][1]} | {compare_target[i][2]}',
            fontsize=15,
            multialignment='center'
        )
        plt.xlabel('\n'.join(wrap(compare_target[i][0], 40)), multialignment='center')
        plt.tight_layout()

Посмотрим на случаи, когда краудсорсинг давал оценку 0, а эксперты больше 0.6 по медиане

In [ ]:
image_comparison(df_compare.query('crowd_mean_score==0.0 & exp_median_score>0.6'))

- Экспертная оценка показывает наличие частичного пересечения запроса и контекста изображения, например
    - в запросе - `Большая собака играет с двумя маленькими собаками на траве`
    - на изображении - `Одна или две собаки играют на траве`
    - краудсорс ставит 0, а эксперты ставят 0.67 по медиане или 0.56 / 0.67 среднюю оценку. Причем в средней оценке у экспертов 0.56 для одной собаки, играющией на траве, и 0.67 для двух, что говорит в пользу выбора средней экспертной оценки как более точно отражающей контекст

Посмотрим на пары, где краудсорс поставил 1, а эксперты меньше 1

In [ ]:
image_comparison(df_compare.query('crowd_mean_score==1.0 & exp_median_score<1'))

- Много случаев, когда краудсорс будто правильно ставит 1, хотя возможно им не выдавался более точный вариант запроса в выборке, а возможно показывали только 1 пару запроса-изображения. С одной стороны, в боевом варианте такой подход оценки краудсорсинга будет лучше, с другой все будет зависеть от количества пар, не хватит экспертов на все это дело и надо будет что-то придумывать еще, например пользователь при загрузке изображения должен описать, что на ней происходит
- Средняя оценка экспертов лучше отражает контекст, чем медианная
- Процент, когда нельзя согласиться с краудсорсерами уже не такой большой

Посомотрим, наконец на совпадения оценок краудсорса и экспертов

In [ ]:
image_comparison(df_compare.query('crowd_mean_score==exp_median_score & crowd_mean_score>0.0'))

- Тут в целом все понятно, средняя оценка экспертов опять как будто лучше подходит к контексту изображения
- Можно брать среднюю оценку экспертов для пар, где есть и оценки краудсорсинга, а там где нет - что-то одно
- Можно взять только среднюю оценку экспертов
- Посмотрим просто по результатам моделей на этапе обучения, а пока сделаем два варианта датасетов

In [ ]:
df_train.drop('exp_median_score', axis=1, inplace=True)

### Возрастные юридические ограничения

Подготовим текстовые запросы

- Почистим от спецсимволов и приведем к нижнему регистру
- Лемматизируем для поиска запросов, содержащих упоминания людей до 16 лет

In [ ]:
df_query_prep = df_train[['query_id', 'query_text']].drop_duplicates()

In [ ]:
nlp = spacy.load('en_core_web_sm')
stopwords = nlp.Defaults.stop_words

In [ ]:
def cleaner(text):
    # Убираем интернет-ссылки, спецсимволы, оставляем только текст и приводим к нижнему регистру
    text = re.sub(r'(?:(?:https?|ftp):\/\/)?[\w/\-?=%.]+\.[\w/\-&?=%.]+|\d+', '', text)
    text = re.sub(r'[^ a-zA-Z]+', '', text)

    return text.lower()

In [ ]:
df_query_prep['query_text'] = df_query_prep['query_text'].progress_apply(cleaner)

In [ ]:
def lemmatize_pipe(doc):
    lemma_list = [str(tok.lemma_).lower() for tok in doc
                  if tok.is_alpha and tok.text.lower() not in stopwords]
    return lemma_list

def preprocess_pipe(texts):
    preproc_pipe = []
    for doc in tqdm(nlp.pipe(texts, n_process=-1),
                    total=len(texts)):
        preproc_pipe.append(lemmatize_pipe(doc))
    return preproc_pipe

In [ ]:
df_query_prep['lemma_q_text'] = preprocess_pipe(df_query_prep['query_text'])

In [ ]:
minor_set = set([
    'adolescent',
    'baby',
    'babies',
    'boy',
    'child',
    'children',
    'girl',
    'infant',
    'junior',
    'juvenile',
    'newborn',
    'teenager',
    'youngster',
    'youth',
    'lad',
    'schoolboy',
    'schoolgirl',
    'pupil',
    'underage',
    'kid',
    'kiddie',
    'youngling',
    'toddler',
    'teen',
    'juvenile',
    'adolescent',
    'kiddy',
    'teenager',
    'youngster',
    'tween',
    'moppet',
    'kiddo',
    'youth',
    'classmate'
])

df_query_prep['minor_restrict'] = df_query_prep['lemma_q_text'].apply(
    lambda x: True if minor_set.intersection(set(x)) else False
)

In [ ]:
display(df_query_prep.head())
print('Записей, поподающих под ограничения - ',
      df_query_prep.query('minor_restrict==True').shape[0])

Теперь получим список изображений, которые содержат изображения детей и подростков. Выше мы отмечали, что запрос `query_id` строится из названия фотографии + `#2`, поэтому просто у всех подходящих под ограничения запросов уберем суффикс и получим список изображений

In [ ]:
restrict_images = df_query_prep.query('minor_restrict==True')['query_id'].apply(
    lambda x: x[:-2]
)
print('Изображений с людьми до 16 лет - ', len(restrict_images))

Уберем все пары содержащие эти изображения

In [ ]:
df_train = df_train.query('image not in @restrict_images')
df_crowd = df_crowd.query('image not in @restrict_images')
df_train = (
    df_train
    .drop('query_text', axis=1)
    .merge(df_query_prep[['query_id', 'query_text']], on='query_id', how='left')
)
df_train.info()
print('\nУникальных изображений - ', df_train['image'].nunique())

In [ ]:
df_crowd.info()
print('\nУникальных изображений - ', df_crowd['image'].nunique())

- Отлично, осталось 4067 пар изображний-запрос и 707 уникальных изображений в `df_train`
- 33372 пары и 707 уникальных изображений в `df_crowd`

### Создание общей выборки

In [ ]:
df_train_ext = pd.merge(
    df_crowd[['image', 'query_id', 'crowd_mean_score']],
    df_train[['image', 'query_id', 'exp_mean_score']],
    on=['image', 'query_id'],
    how='outer',
).merge(df_query_prep[['query_id', 'query_text']], on='query_id', how='left')
df_train_ext.info()

Отлично, получилось два датасета
- `df_train` - только с экспертными оценками
- `df_train_ext` - с оценками экспертов и краудсорсинга, сделаем из них один таргет (экспертная оценка если есть, если она отсутствует, то краудсорс оценка)

In [ ]:
df_train_ext['score'] = df_train_ext['exp_mean_score'].where(
    ~df_train_ext['exp_mean_score'].isna(), other=df_train_ext['crowd_mean_score'])
df_train_ext.drop(['crowd_mean_score', 'exp_mean_score'], axis=1, inplace=True)
df_train_ext.info()

In [ ]:
df_train.rename(columns={'exp_mean_score': 'score'}, inplace=True)

In [ ]:
del df_crowd

### Вывод

- В целом эксперты более менее согласованны, хотя есть редкие случаи вроде (1, 3, 3) или (1, 4, 4) например, один из них чаще завышает, другой занижает, а третий между ними
- В качестве агрегации оценок выбрали среднее арифметическое из моды, медианы и среднего
- Краудсорсинг заметно занижает оценки соответствия пар по сравнению с экспертами. Сделали два варианта датасетов
    - `df_train` - только с экспертными оценками - 4067 пар
    - `df_train_ext` - с оценками экспертов и краудсорсинга, (экспертная оценка если есть, если она отсутствует, то краудсорс оценка) - 35000 пар
- Убрали 293 изображения  и соответствующих запросов из тренировочных данных, в соответствии с юридическими ограничениями

## Подготовка данных

### Векторизация текстов

- Возьмем последний скрытый слой из BERT модели
- Так как тексты разной длины, он доводятся паддинггом до равной длины, но для обучения под нашу задачи 0 паддинга будут мешать, поэтому на каждый эмбеддинг применим маску, чтобы не учитывать 0 паддинга

In [ ]:
bert_model = TFBertModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def get_text_embedding(text, max_len=64):
    tokens = tokenizer(text, padding='max_length', truncation=True, max_length=max_len, return_tensors='tf')
    input_ids = tokens.input_ids
    attention_mask = tf.cast(tokens.attention_mask, dtype=tf.float32)

    outputs = bert_model(input_ids, attention_mask=tokens.attention_mask, training=False)
    last_hidden = outputs.last_hidden_state

    mask = tf.expand_dims(attention_mask, axis=-1)
    masked_hidden = last_hidden * mask

    sum_hidden = tf.reduce_sum(masked_hidden, axis=1)
    counts = tf.reduce_sum(mask, axis=1)
    emb = sum_hidden / counts
    return tf.squeeze(emb, axis=0)  # (768)

In [ ]:
def df_text_to_emb(df):
    unique_texts = df['query_text'].unique()
    text_embedding_map = {
        text: get_text_embedding(text).numpy()
        for text in unique_texts
    }
    df['text_emb'] = df['query_text'].map(text_embedding_map)

In [ ]:
df_text_to_emb(df_train)

### Векторизация изображений

Здесь возьмем последний скрытый слой из ViT модели, в ней используются патчи, что по идее должно помочь при сведении векторов

In [ ]:
model_name = 'google/vit-base-patch16-224-in21k'
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)
vit_model = TFViTModel.from_pretrained(model_name)

In [ ]:
def get_image_embedding(img_path):
    image = Image.open(img_path).convert("RGB")
    inputs = feature_extractor(images=image, return_tensors="tf")
    outputs = vit_model(**inputs)
    cls_embedding = outputs.last_hidden_state[:, 0]
    return cls_embedding.numpy().squeeze()

In [ ]:
def df_image_to_emb(df, folder):
    unique_images = df['image'].unique()
    image_embedding_map = {
        image: get_image_embedding(os.path.join(DATA_PATH, folder, image))
        for image in unique_images
    }
    df['image_emb'] = df['image'].map(image_embedding_map)

In [ ]:
df_image_to_emb(df_train, TRAIN_IMAGES_DIR)

Отлично, векторизовали тексты и изображения с помощью BERT и ViT, используя их бэкбоны, теперь будем пробовать сводить эти вектора вместе с помощью модели

### Создание обучающей и валидационной выборки

In [ ]:
gss = GroupShuffleSplit(n_splits=3, train_size=0.8, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss.split(df_train, y=df_train['score'], groups=df_train['image']))
train_df = df_train.iloc[train_idx].reset_index(drop=True)
val_df = df_train.iloc[val_idx].reset_index(drop=True)

X_text_train = np.stack(train_df['text_emb'].values)
X_image_train = np.stack(train_df['image_emb'].values)
y_score_train = train_df['score'].values.astype(np.float32)

X_text_val = np.stack(val_df['text_emb'].values)
X_image_val = np.stack(val_df['image_emb'].values)
y_score_val = val_df['score'].values.astype(np.float32)

### Вывод

- Векторизовали тексты и изображения с помощью BERT и ViT последними скрытыми слоями моделей
- Из текстового эмбеддинга убрали влияние нулей паддинга маской

## Подбор и обучение моделей

- Будем сравнивать модели по RSME разницы косинусого расстояния и `score`
- Для бейзлайна возьмем Dummy Regressor

### Dummy

In [ ]:
X_train = np.concatenate([X_text_train, X_image_train], axis=1)
X_val = np.concatenate([X_text_val, X_image_val], axis=1)

model_scaler = MinMaxScaler()

X_train_scaled = model_scaler.fit_transform(X_train)
X_valid_scaled = model_scaler.transform(X_val)

In [ ]:
dummy = DummyRegressor()
dummy.fit(X_train, y_score_train)
rmse_dummy_train = root_mean_squared_error(y_score_train, dummy.predict(X_train))
rmse_dummy_val = root_mean_squared_error(y_score_val, dummy.predict(X_val))
print(f'RMSE Dummy модели на обучении - {rmse_dummy_train:.4f}')
print(f'RMSE Dummy модели на валидации - {rmse_dummy_val:.4f}')

Отлично, бейзлайн в 0.28 есть, теперь можно будет хоть как-то оценивать обучение моделей

### Linear Regression

In [ ]:
lr = LinearRegression(fit_intercept=False)
lr.fit(X_train_scaled, y_score_train)
rmse_lr_train = root_mean_squared_error(y_score_train, lr.predict(X_train))
rmse_lr_val = root_mean_squared_error(y_score_val, lr.predict(X_val))
print(f'RMSE линейной регрессии на обучении - {rmse_lr_train:.4f}')
print(f'RMSE линейной регрессии на валидации - {rmse_lr_val:.4f}')

Линейная регрессия заметно хуже бейзлайна, а на валидации вообще просела, как будто переобучение. По идее оно возникло из-за того, что эмбединги длиной 768 для текстов и изображений, то есть 1536 признаков, причем они все нормированы, и значит наверняка есть зависимости между признаками, корреляции

### Text-Image Retrieval NN

Попробуем собрать нейронную архитектуру. Сделаем функцию проекционной головы, энкодеры для векторов текста и изображения, чтобы на инференсе было проще получать вектора. По-хорошему надо бы сделать хороший загрузчик батча, где у нас одному текстовому запросу шли вместе связанные изображение по таблице и оценки их соответствия, и тогда можно было считать soft contrastive loss. Функцию для лосса легко реализовать, а вот с батчами трудновато, чтобы еще по ресурсам не выйти за возможности

In [ ]:
def deep_projection_head(inputs, output_dim=256, dropout_rate=0.2, num_heads=2):

    out = layers.Dense(output_dim)(inputs)
    for i in range(num_heads):
        x = layers.Activation('gelu')(out)
        x = layers.Dense(output_dim)(x)
        x = layers.Dropout(dropout_rate)(x)
        x = layers.Add()([out, x])
        out = layers.LayerNormalization()(x)

    return out

In [ ]:
def create_vision_encoder(
    num_heads, projection_dims, dropout_rate, trainable=False, embedding_dim_image=768
):
    image_input = Input(shape=(embedding_dim_image,), name='image_input')
    output = deep_projection_head(
        image_input, projection_dims, dropout_rate, num_heads
    )
    return Model(image_input, output, name='vision_encoder')

In [ ]:
def create_text_encoder(
    num_heads, projection_dims, dropout_rate, trainable=False, embedding_dim_text=768
):
    text_input = Input(shape=(embedding_dim_text,), name='text_input')
    output = deep_projection_head(
        text_input, projection_dims, dropout_rate, num_heads
    )
    return Model(text_input, output, name='text_encoder')

In [ ]:
def build_model(
    text_encoder, vision_encoder, embedding_dim_image=768, embedding_dim_text=768
):
    text_input = Input(shape=(embedding_dim_text, ), name='text_input')
    image_input = Input(shape=(embedding_dim_image, ), name='image_input')
    with tf.device('/gpu:0'):
        text_proj = text_encoder(text_input)
    with tf.device('/gpu:1'):
        image_proj = vision_encoder(image_input)
    cosine_sim = layers.Dot(axes=1, normalize=True)([text_proj, image_proj])
    sim_scaled = layers.Lambda(lambda x: (x + 1) / 2)(cosine_sim)

    return Model(inputs=[text_input, image_input], outputs=(sim_scaled))

In [ ]:
vision_encoder = create_vision_encoder(num_heads=1,
                                       projection_dims=768,
                                       dropout_rate=0.2,
                                       embedding_dim_image=768)
text_encoder = create_text_encoder(num_heads=1,
                                   projection_dims=768,
                                   dropout_rate=0.2,
                                   embedding_dim_text=768)
NN_model = build_model(text_encoder,
                       vision_encoder,
                       embedding_dim_image=768,
                       embedding_dim_text=768)

NN_model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss='mse',
    metrics=['root_mean_squared_error']
)

In [ ]:
NN_model.fit(
    x=[X_text_train, X_image_train],
    y=y_score_train,
    validation_data=([X_text_val, X_image_val], y_score_val),
    batch_size=64,
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    ]
)

RMSE на валидации около 0.228, лучше чем Dummy модель, но все равно многовато скорее всего. Из множества разных попыток получалось достичь 0.2 на другой архитектуре и InceptionResNetV2 вместо ViT

### Инференс

In [ ]:
test_images_emb = df_test['image'].unique()
image_embeddings = [get_image_embedding(os.path.join(DATA_PATH, TEST_IMAGES_DIR, image))
                    for image in test_images_emb]
image_embeddings = np.stack(image_embeddings)

In [ ]:
image_proj_matrix = vision_encoder.predict(image_embeddings, batch_size=64, verbose=0)
image_proj_matrix = image_proj_matrix / np.linalg.norm(image_proj_matrix, axis=1, keepdims=True)

In [ ]:
def find_top_k_matches(query_text, top_k=5, clip=False):

    print(f'Введенная фраза:{query_text} \n')
    query_text = cleaner(query_text)
    text_lemm = lemmatize_pipe(nlp(query_text))
    for word in text_lemm:
        if word in minor_set:
            return print('This image is unavailable in your country in compliance with local laws\n')

    if clip==False:
        text_emb = get_text_embedding(query_text).numpy().reshape(1, -1)
        text_proj = text_encoder.predict(text_emb, verbose=0)
    else:
        text_proj = get_clip_text_emb(query_text)
    text_proj = text_proj / np.linalg.norm(text_proj)


    sims = np.dot(image_proj_matrix, text_proj.T).flatten()
    sims = (sims + 1) / 2

    # Топ-k индексов по убыванию
    top_indices = np.argsort(sims)[::-1][:top_k]
    top_matches = [(test_images_emb[i], sims[i]) for i in top_indices]

    # Визуализация
    n_cols = min(top_k, 5)
    n_rows = (top_k + n_cols - 1) // n_cols

    plt.figure(figsize=(n_cols * 3, n_rows * 3))
    plt.suptitle(f'Запрос: "{query_text}"', fontsize=16)

    for idx, (filename, sim) in enumerate(top_matches):
        img_path = os.path.join(DATA_PATH, TEST_IMAGES_DIR, filename)
        img = Image.open(img_path)

        plt.subplot(n_rows, n_cols, idx + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"{sim:.2f}", fontsize=10)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

In [ ]:
def evaluate_model_on_random_queries(df_test, top_k=5, num_queries=10, clip=False):
    queries = df_test['query_text'].unique()
    sampled_queries = random.sample(list(queries), num_queries)
    for query in sampled_queries:
        find_top_k_matches(query, top_k=top_k, clip=clip)

In [ ]:
evaluate_model_on_random_queries(df_test, top_k=5, num_queries=10)

Даа, вышло так себе, попробуем предобученный CLIP, потому что такой вариант заказчику не стоит показывать

### CLIP

Посмотрим на предобученную CLIP модель

In [ ]:
clip_model = TFCLIPModel.from_pretrained('openai/clip-vit-base-patch32')
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

In [ ]:
def get_clip_text_emb(text):
    inputs = clip_processor(text=[text], return_tensors='tf')
    outputs = clip_model.get_text_features(**inputs)
    return outputs.numpy().squeeze()

def get_clip_image_emb(img_path):
    image = Image.open(img_path).convert('RGB')
    inputs = clip_processor(images=image, return_tensors='tf')
    outputs = clip_model.get_image_features(**inputs)
    return outputs.numpy().squeeze()

In [ ]:
image_proj_matrix = [get_clip_image_emb(os.path.join(DATA_PATH, TEST_IMAGES_DIR, image))
                        for image in test_images_emb]
image_proj_matrix = np.stack([vec / np.linalg.norm(vec) for vec in image_proj_matrix])

In [ ]:
evaluate_model_on_random_queries(df_test, top_k=5, num_queries=10, clip=True)

Да, этот вариант явно лучше. Но необходимо заметить, что хоть мы и сделали юридический фильтр по запросу, изображения с детьми проскакивают, значит заказчику надо будет рекомендовать подумать об отдельной функции на этот счет

## Общий вывод

В рамках проекта была создана модель поиска изображений по текстовому запросу на основе **BERT** и **ViT**, которая показала метрику **RMSE 0.228** на валидации, но по факту оказалось не достаточно, для улучшения результатов была использована предобученная модель CLIP, которая показывает хорошие результаты.

**Входные данные**
- Всего записей
    - `df_train` - 5822
    - `df_expert` - 5822
    - `df_crowd` - 47830
- В каждом 1000 уникальных объектов изображений

**Исследовательский анализ**
- В целом эксперты более менее согласованны, хотя есть редкие случаи вроде (1, 3, 3) или (1, 4, 4) например, один из них чаще завышает, другой занижает, а третий между ними
- В качестве агрегации оценок выбрали среднее арифметическое из моды, медианы и среднего
- Краудсорсинг заметно занижает оценки соответствия пар по сравнению с экспертами. Сделали два варианта датасетов
    - `df_train` - только с экспертными оценками - 4067 пар
    - `df_train_ext` - с оценками экспертов и краудсорсинга, (экспертная оценка если есть, если она отсутствует, то краудсорс оценка) - 35000 пар
- Убрали 293 изображения  и соответствующих запросов из тренировочных данных, в соответствии с юридическими ограничениями

**Text-Image Retrieval модель**
- RMSE на валидации
    - **Baseline** Dummy модель - 0.296
    - Linear Regression - 0.58
    - Custom NN - 0.228

- В кастомной NN использовали проекционные головы, энкодеры для векторов текста и изображения
- На инференсе показала себя плохо
- Вероятно помогло бы кастомный загрузчик батчей, где у нас одному текстовому запросу шли вместе связанные изображение по таблице и оценки их соответствия
- Soft Contrastive Loss
- Для быстрого решения вопроса была использована предобученная CLIP модель от OpenAI, которая показала отличные результаты
- Ее так же можно дообучить на нашей выборке для дальнейшей работы, для текущей задачи демонстрационного варианта она подходит и так

**Возможные улучшения модели**
- Кастомный загрузчик батча и данных, скорее всего кастомный трейнер
- Soft Contrastive Loss
- Как вариант дообучить CLIP

**Рекомендации заказчику и комментарии по поводу предоставленной выборки**
- Повысить надежность краудсорса, возможно через вознаграждение, чтобы точность оценок была выше и можно было дообучать модель
- Добавить возможность или как обязательный вариант, при загрузке пользователем изображения на сайт - обязательно заполнить краткое описание
- Фильтрацию по изображениям в юридическом контексте сделать отдельно, так как только текстовой фильтрации недостаточно, если реализовать функцию выдачи топ-N изображений по запросу, модель может выдать изображение, где будут дети